# 面向长对话的贝叶斯自适应上下文选择 · Release研究任务工作本
**ARIN7101 · release-2.1 scaffold · 2026-11-05**

研究问题：在受控合成长对话中，利用问题特征的贝叶斯策略选择能否改善正确性与输入成本的取舍？

**这是任务脚手架，研究实现和结果评估由成员自行完成。** 这里只提供公共路径、实验设计草案、接口骨架与任务提示。研究函数均以NotImplementedError留空；第0节会检查环境、建立统一目录并保存环境记录；M1–M5仍只定义骨架，不生成研究数据、统计或图，初始化成功不代表任务完成。

先读[README](README.md)和[协作说明](docs/TEAM_WORKFLOW.md)。每个人须交付自己的代码、自己设计的验证、实际观察与局限和可追踪产物。参考完整本只作可行性背景，release不从中导入或执行实现；旧数值、图表与结论不属于本组已完成的研究。

| 成员 | 必须自己研究、编码的范围 | cell ID区间 | 接收人 |
|---|---|---|---|
| [M1](#member-m1) | 数据生成、公开信息边界、独立真值核查 | m1-* | M2/M4 |
| [M2](#member-m2) | 四策略、回答器、成本、评价及反例 | m2-* | M1/M3/M4 |
| [M3](#member-m3) | 特征、TS/CTS、后验数值验证、回放 | m3-* | M4 |
| [M4](#member-m4) | 开发选择、冻结、实验调度、统计与归档 | m4-* | M3复核/M5 |
| [M5](#member-m5) | 读取校验、可视化编码、结果解释与汇报 | m5-* | 全组 |


## 0 · 环境配置及路径约定
**本节代码已写好，全员首次下载仓库后直接按顺序运行本节两个代码单元。** 它负责依赖检查、公共导入、根目录定位、统一目录初始化、环境记录及安全保存工具。之后各成员使用这里的路径变量；无需自行拼接绝对路径。

### 0.1 · 首次准备与环境检查
建议Python 3.11或3.12。在仓库根目录创建虚拟环境并安装Jupyter（首次尚无法打开notebook时执行）：

    python -m venv .venv
    # Windows PowerShell：.\.venv\Scripts\Activate.ps1
    # macOS/Linux：source .venv/bin/activate
    python -m pip install -r requirements.txt
    python -m ipykernel install --user --name arin7101 --display-name "Python 3 (ARIN7101)"
    python -m jupyter lab

选择上述内核。下一单元使用当前内核的Python检查requirements；若依赖缺失或版本不符，默认自动安装，并要求重启内核后重跑，以免混用已加载的旧包。无需GPU、API密钥或外部数据。

PROJECT_ROOT默认自动从当前目录向上寻找仓库；若从仓库外启动内核，将它填为本机仓库根路径。不要把自己的绝对路径提交Git。AUTO_INSTALL_MISSING可改为False，只检查并给出安装提示。


In [ ]:
# 0.1：全员直接运行。只处理环境与公共导入。
import sys
import os
import re
import json
import platform
import subprocess
from pathlib import Path
from datetime import datetime
from importlib import metadata
from packaging.requirements import Requirement
from IPython.display import display

PROJECT_ROOT = None  # 通常无需填写；仓库外启动时可填本机路径
AUTO_INSTALL_MISSING = True

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("本项目统一使用Python 3.11或3.12；请切换对应Jupyter内核。")
_candidates = ([Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT
               else [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in _candidates
             if (p / "ARIN7101_Context_Bandit_Release_Notebook.ipynb").is_file()
             and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("未定位仓库，请填写PROJECT_ROOT或从仓库内启动Jupyter。")

_requirements = []
for _line in (ROOT / "requirements.txt").read_text(encoding="utf-8-sig").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if _line:
        _req = Requirement(_line)
        if _req.marker is None or _req.marker.evaluate():
            _requirements.append(_req)

_problems = []
for _req in _requirements:
    try:
        _version = metadata.version(_req.name)
    except metadata.PackageNotFoundError:
        _problems.append(f"{_req.name} 未安装")
    else:
        if _version not in _req.specifier:
            _problems.append(f"{_req.name}={_version}，要求 {_req.specifier}")
if _problems:
    print("需要处理的依赖：\n- " + "\n- ".join(_problems))
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError(f'请用当前内核Python安装依赖："{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"')
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
                   check=True)
    raise RuntimeError("依赖已安装。请Restart Kernel，重新运行第0节，避免混用已加载的旧包。")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "executable": sys.executable,
    "packages": {_req.name: metadata.version(_req.name) for _req in _requirements},
}
print("环境检查通过")
print("项目根目录：", ROOT)
print("当前内核：", sys.executable)
display(pd.DataFrame(list(ENVIRONMENT["packages"].items()), columns=["package", "version"]))


### 0.2 · 初始化工作目录和公共路径
运行下一单元，默认建立本机练习目录outputs/development/local_setup及其六个子目录，并写入config/environment.json。已有文件与人工记录不覆盖。重复执行同一个development/demo配置可复用目录；正式run必须是新目录。

**每次实际实验开始前修改MODE和RUN_ID**，例如development / m1_trial01，或demo / team_integration01。成员交接必须明确run路径；同名run只有在配置、输入与来源一致时才能合用。正式运行由M4完成审查与冻结后初始化，初始化不等于获得正式执行资格。

| 路径/变量 | 用途与负责成员 | 本节行为 | Git |
|---|---|---|---|
| ROOT | 仓库根目录，自动定位 | 查找，不改工作目录 | — |
| TEAM_DIR = team_workspace/ | 姓名、进度、接口决策、正式尝试登记；全员/M4 | 确保目录存在，不改人工记录 | 提交 |
| DELIVERABLES_DIR = deliverables/ | 个人研究说明、最终slides、提交清单；全员 | 确保目录存在 | 提交小文件，视频用链接 |
| DOCS_DIR / REFERENCE_DIR | 协作规范、原计划、历史参考 | 仅定位/确保目录存在 | 提交 |
| OUTPUTS_DIR = outputs/ | 自动产物总目录 | 建立development/demo/formal三个分区 | 忽略 |
| RUN_DIR = outputs/MODE/RUN_ID/ | 本次开发/实验的唯一根目录 | 初始化；formal禁止复用已有目录 | 忽略 |
| DATA_DIR = RUN_DIR/data/ | 生成对话与原始数据；M1/M4 | 创建空目录 | 忽略 |
| TABLES_DIR = RUN_DIR/tables/ | 开发评价、回放、统计表、绘图数据；M2–M5 | 创建空目录 | 忽略 |
| AUDIT_DIR = RUN_DIR/audit/ | 真值/策略/方法检查、十题审查与证据副本；M1–M4 | 创建空目录 | 忽略 |
| CONFIG_DIR = RUN_DIR/config/ | 环境记录、冻结配置、代码快照；公共设施/M4 | 写环境记录，其余由成员完成 | 忽略 |
| FIGURES_DIR = RUN_DIR/figures/ | 自己编写代码生成的PNG/SVG；M5 | 创建空目录 | 忽略 |
| REPORTS_DIR = RUN_DIR/reports/ | 结果交接、逐图解读、图表清单；M4/M5 | 创建空目录 | 忽略 |
| RUN_DIR/manifest.json | 整个实验的来源与文件校验清单；M4 | 不生成，留给M4实现 | 忽略 |

**统一使用方式：**
- 自己实现的函数接收RUN_DIR；单文件用artifact_path(MODE, RUN_ID, area, filename)取路径。
- 已提供save_json(path, value)和save_csv(path, dataframe)；它们限定outputs内部、UTF-8编码，拒绝覆盖已有文件。不再让每人重复写路径/通用保存代码。
- 绘图、压缩对话等专用导出仍由成员实现，使用上述目录。JSON、CSV的内容与研究指标仍由成员计算。
- M5读取M4交付的明确路径作为SOURCE_RUN，不重新初始化/改写上游formal目录；可以另用自己的development run试绘。
- 本节只有环境信息，不生成研究配置冻结、数据、统计、图或成员签名。使用新RUN_ID保留修改前产物；不要以重跑初始化来清理旧结果。


In [ ]:
# 0.2：全员直接运行；做实际实验时改这两个值，不改下面的统一目录规则。
MODE = "development"     # development / demo / formal
RUN_ID = "local_setup"   # 正式实验由M4在冻结后使用全新ID

TEAM_DIR = ROOT / "team_workspace"
DELIVERABLES_DIR = ROOT / "deliverables"
DOCS_DIR = ROOT / "docs"
REFERENCE_DIR = ROOT / "reference"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUT_AREAS = ("data", "tables", "audit", "config", "figures", "reports")
MODES = ("development", "demo", "formal")

def _safe_component(value):
    if (not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value)
            or value.endswith(".") or value.split(".", 1)[0].upper() in
            {"CON", "PRN", "AUX", "NUL", *[f"COM{i}" for i in range(1, 10)],
             *[f"LPT{i}" for i in range(1, 10)]}):
        raise ValueError("名称须以字母/数字开头，只含字母、数字、下划线、连字符和点；不可用系统保留名。")
    return value

def _run_path(mode, run_id):
    if mode not in MODES:
        raise ValueError(f"mode须为{MODES}")
    path = OUTPUTS_DIR / mode / _safe_component(run_id)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def artifact_path(mode, run_id, area, filename):
    """返回统一文件路径；先使用init_run初始化该run，不在此处生成研究结果。"""
    if area not in OUTPUT_AREAS:
        raise ValueError(f"area须为{OUTPUT_AREAS}")
    path = _run_path(mode, run_id) / area / _safe_component(filename)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def _writable_output(path):
    path = Path(path).resolve()
    if not path.is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("通用保存工具只写outputs；人工记录请本人编辑。")
    if not path.parent.is_dir():
        raise FileNotFoundError("输出目录不存在；请先init_run。")
    return path

def save_json(path, value):
    """统一UTF-8/严格JSON，拒绝覆盖；不代替研究计算或签名。"""
    payload = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
    with _writable_output(path).open("x", encoding="utf-8") as stream:
        stream.write(payload)

def save_csv(path, dataframe):
    """保存成员计算得到的DataFrame；统一UTF-8、无索引、拒绝覆盖。"""
    dataframe.to_csv(_writable_output(path), index=False, encoding="utf-8", mode="x")

def init_run(mode, run_id):
    """创建六个目录和环境记录；开发/demo可复用，formal禁止复用。"""
    run_dir = _run_path(mode, run_id)
    if mode == "formal" and run_dir.exists():
        raise FileExistsError("正式run目录已存在；请保留原目录，不能重跑初始化覆盖。")
    # 先检查既有环境，不改已有记录；环境变化应使用新的run_id。
    env_path = run_dir / "config" / "environment.json"
    if env_path.exists():
        existing = json.loads(env_path.read_text(encoding="utf-8"))
        if (existing.get("python"), existing.get("packages")) != (
                ENVIRONMENT["python"], ENVIRONMENT["packages"]):
            raise RuntimeError("本run已记录不同Python/依赖版本；请用新RUN_ID保留环境来源。")
    run_dir.mkdir(parents=True, exist_ok=(mode != "formal"))
    for area in OUTPUT_AREAS:
        directory = run_dir / area
        if not directory.resolve().is_relative_to(run_dir.resolve()):
            raise ValueError("子目录不能指向run以外的位置。")
        directory.mkdir(exist_ok=True)
    if not env_path.exists():
        save_json(env_path, {
            **ENVIRONMENT, "mode": mode, "run_id": run_id,
            "created_at": datetime.now().astimezone().isoformat(),
            "status": "environment_initialized_only",
        })
    return run_dir

for directory in (TEAM_DIR, DELIVERABLES_DIR, DOCS_DIR, REFERENCE_DIR, OUTPUTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for mode in MODES:
    (OUTPUTS_DIR / mode).mkdir(exist_ok=True)

RUN_DIR = init_run(MODE, RUN_ID)
RUN_PATHS = {area: RUN_DIR / area for area in OUTPUT_AREAS}
DATA_DIR = RUN_PATHS["data"]
TABLES_DIR = RUN_PATHS["tables"]
AUDIT_DIR = RUN_PATHS["audit"]
CONFIG_DIR = RUN_PATHS["config"]
FIGURES_DIR = RUN_PATHS["figures"]
REPORTS_DIR = RUN_PATHS["reports"]

print("初始化完成；未生成研究结果。")
print("RUN_DIR =", RUN_DIR)
display(pd.DataFrame(
    [{"area": area, "relative_path": directory.relative_to(ROOT).as_posix(),
      "absolute_path": str(directory)} for area, directory in RUN_PATHS.items()]
))
print("环境记录：", CONFIG_DIR / "environment.json")

# 在成员完成计算后使用（这里只示范路径，不生成假的结果文件）：
# save_csv(artifact_path(MODE, RUN_ID, "tables", "development_outcomes.csv"), outcomes)
# save_json(artifact_path(MODE, RUN_ID, "audit", "truth_checks.json"), checks)
# M5使用明确的SOURCE_RUN读取M4交付物，不调用init_run去重建上游formal目录。


## 共同研究设计与成员探索空间
以下设计草案供M1–M5研究；与第0节的环境初始化分开，确认冻结由M4负责。

**原计划提供的设计起点，开发结束前由全组确认冻结：**
- 6个项目、4种属性，英文模板历史含focus、事实更新、旧版本与干扰；每对话300题。
- stable的近期题概率0.9；abrupt在第151题起从0.9变0.1。改变的是提问需求分布。
- recent_short：最近最多3条，预算128；recent_long：连续近期窗口，512；tfidf：原问题检索最多8个候选，256；resolved_tfidf：用公开历史消解指代/同义词后检索，256。
- U = Y − λC/Cref，λ=0.5、Cref=512。C是整个输入的正则长度代理，不是真实token费用；预算仅限制所选上下文。
- 开发每场景3条、demo每场景6条、正式每场景30条。TS/CTS算法seed为17/37，每对话重置。
- 主比较：CTS与各场景开发集选定的固定臂；TS为次比较；RuleRouter为模板规则参照。算法seed先在对话内平均，再按独立对话配对bootstrap，报告95%和两主比较校正97.5%区间。

**成员需要自行决定并研究：** 具体模板与版本生成细节、检索平局、数值优化、失败处理、测试案例与图型选择。记录理由、比较和局限，不能只复制完整本。对共同设计的修改须在正式评估前确认；测试结果之后的修改另标探索/修正。

**并行起步：** M1先做可追踪小历史；M2用自建历史；M3用自建四臂小表；M4设计统计手算案例；M5用明确标为fixture的虚构表研究绘图。示意数据不进入真实结果。

**接通顺序：** M1生成器 → M2评价器 → M1/M2共同审计 → M3回放 → M4集成与冻结 → 正式运行 → M5解释。未实现函数被调用时明确报错，不用空表/固定返回值掩盖未完成。


In [ ]:
ARMS = ("recent_short", "recent_long", "tfidf", "resolved_tfidf")
SCENARIOS = ("stable", "abrupt")
METHODS = ARMS + ("TS", "LaplaceCTS", "RuleRouter")
PROTOCOL_DRAFT = {
    "T": 300, "n_dev_per_scenario": 3, "n_demo_per_scenario": 6,
    "n_formal_per_scenario": 30, "dev_seed": 130000, "demo_seed": 730000,
    "formal_seed": None,  # M4与全组确认未使用的seed后填写
    "algorithm_seeds": [17, 37], "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9, "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25, "initial_conflict_fraction": 0.25,
    "budgets": dict(zip(ARMS, [128, 512, 256, 256])),
    "lambda": 0.5, "cost_reference": 512, "refit_every": 5,
    "prior_precision": 1.0, "bootstrap_seed": 2026, "n_bootstrap": 2000,
}


### 共同接口
详细字段见[输出契约](docs/OUTPUT_CONTRACT.md)。骨架参数名是交接约定；模块内部实现由成员选择。
- episode：seed/scenario/history/probes；message：id/text；probe：t/question/end/gold/evidence/old_values/qtype/state。
- public_view仅含question/history；gold、隐藏题型、未选反馈只供评估器访问。
- 四臂表：seed,scenario,t,arm,question,y,cost,utility,coverage,outdated,construction_ms,qtype,state。
- 回放表：seed,scenario,t,method,alg_seed,arm,y,cost,utility,coverage,outdated。
- 对话指标：scenario,seed,method,y,cost,utility,coverage,outdated。
- 主效应表：scenario,method,baseline,n,effect,low,high,family_low,family_high。

消费者自行检查字段、唯一键、规模与来源。上游未完成时自行构造明确标为fixture的最小输入，不用完整参考结果填空。


<a id="member-m1"></a>
## M1 · 数据与问题定义
**你要编写生成器、公开视图和独立核查器，研究数据机制。任务区间m1-*。**
输入：共同设计；输出：episode、数据字典、开发数据、核查证据；M2/M4接收。

1. **先设计。** 用纸面例子定义focus/its、对象、属性、最新事实和过期值；决定多版本、复述、同义词如何生成，记录选择理由。
2. **先小后大编码。** 从少量题目的生成器扩展到两个场景与300题；保持当前题只能看历史前缀。
3. **独立核查。** 设计另一条证据追踪路径，不把生成器存的gold读出来就当证明；写复现、指代、更新、旧值、切换边界案例。
4. **交付与联调。** 先给M2微型episode和字典，再导出开发集。M2导出四臂审计后，你追踪10题真值，M2交叉复核。
5. **诊断与解释。** 检查实际题型/版本分布，说明是否人为偏向某种路由；发现问题在开发阶段修改，不预设策略一定胜出。

**验收方向（案例由你设计）：** seed可复现、id连续、无未来证据、真值可追溯、切换正确。接收人能从一条样例找到答案依据。
**输出：** data/development_dialogues.jsonl.gz；audit/data_dictionary.json、truth_checks.json；tables/development_data_profile.csv。使用明确的development run路径。


In [ ]:
import gzip as _m1_gzip
import json as _m1_json
import math as _m1_math
import re as _m1_re
from pathlib import Path as _m1_Path

M1_ATTRIBUTES = ("dataset", "language", "metric", "deadline")
M1_ATTRIBUTE_ALIASES = {
    "dataset": "data source",
    "language": "coding language",
    "metric": "evaluation measure",
    "deadline": "due date",
}
M1_SYSTEM_CONTEXT = (
    "Please use this setting in subsequent work and keep other project requirements unchanged."
)
M1_DEFAULT_CONFIG = {
    "T": 300,
    "n_projects": 6,
    "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9,
    "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25,
    "initial_conflict_fraction": 0.25,
    "alias_probability": 0.5,
}
if "PROTOCOL_DRAFT" in globals():
    M1_DEFAULT_CONFIG.update(PROTOCOL_DRAFT)

_M1_ATTR_PATTERN = "|".join(
    _m1_re.escape(term)
    for term in sorted(
        [*M1_ATTRIBUTES, *M1_ATTRIBUTE_ALIASES.values()],
        key=len,
        reverse=True,
    )
)
_M1_FACT_RE = _m1_re.compile(
    r"^For project (?P<project>P\d+): the (?P<attribute>"
    + _M1_ATTR_PATTERN
    + r") is (?P<value>V\d+)\."
)
_M1_FOCUS_RE = _m1_re.compile(r"^Focus is project (?P<project>P\d+)\.")
_M1_PROJECT_RE = _m1_re.compile(r"\bP\d+\b")
_M1_ALIAS_TO_ATTRIBUTE = {
    term: attribute
    for attribute, alias in M1_ATTRIBUTE_ALIASES.items()
    for term in (attribute, alias)
}


def _m1_config(cfg):
    settings = dict(M1_DEFAULT_CONFIG)
    if cfg is not None:
        settings.update(dict(cfg))
    total = settings["T"]
    projects = settings["n_projects"]
    if isinstance(total, bool) or not isinstance(total, int) or total < 1:
        raise ValueError("cfg['T'] must be a positive integer")
    if isinstance(projects, bool) or not isinstance(projects, int) or projects < 2:
        raise ValueError("cfg['n_projects'] must be an integer of at least 2")
    for name in (
        "p_recent_stable",
        "p_recent_abrupt_before",
        "p_recent_abrupt_after",
        "update_probability",
        "initial_conflict_fraction",
        "alias_probability",
    ):
        value = settings[name]
        if not isinstance(value, (int, float)) or not _m1_math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"cfg[{name!r}] must be a finite probability in [0, 1]")
        settings[name] = float(value)
    return settings


def _m1_canonical_attribute(label):
    return _M1_ALIAS_TO_ATTRIBUTE.get(label.strip().lower())


def _m1_parse_fact(text):
    match = _M1_FACT_RE.match(text)
    if not match:
        return None
    result = match.groupdict()
    result["attribute"] = _m1_canonical_attribute(result["attribute"])
    return result


def _m1_parse_question(question, focus):
    project_match = _M1_PROJECT_RE.search(question)
    project = project_match.group() if project_match else focus
    attribute = None
    for term in sorted(_M1_ALIAS_TO_ATTRIBUTE, key=len, reverse=True):
        if _m1_re.search(r"\b" + _m1_re.escape(term) + r"\b", question, _m1_re.IGNORECASE):
            attribute = _M1_ALIAS_TO_ATTRIBUTE[term]
            break
    return project, attribute, project_match is not None


def generate_episode(seed, scenario, cfg):
    """Generate one reproducible synthetic dialogue using only its local RNG."""
    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or int(seed) < 0:
        raise ValueError("seed must be a non-negative integer")
    seed = int(seed)
    if scenario not in ("stable", "abrupt"):
        raise ValueError("scenario must be 'stable' or 'abrupt'")
    settings = _m1_config(cfg)
    rng = np.random.default_rng(seed)
    projects = [f"P{i + 1}" for i in range(settings["n_projects"])]
    history = []
    probes = []
    facts = {}
    serial = 0

    def append(text):
        message_id = len(history)
        history.append({"id": message_id, "text": text})
        return message_id

    def write_fact(project, attribute, *, update, use_alias):
        nonlocal serial
        key = (project, attribute)
        previous = facts.get(key)
        if previous is None or update:
            serial += 1
            value = f"V{seed * 1_000_000 + serial}"
            versions = [] if previous is None else list(previous["versions"])
            if not versions or versions[-1] != value:
                versions.append(value)
        else:
            value = previous["value"]
            versions = list(previous["versions"])
        label = M1_ATTRIBUTE_ALIASES[attribute] if use_alias else attribute
        evidence = append(
            f"For project {project}: the {label} is {value}. {M1_SYSTEM_CONTEXT}"
        )
        facts[key] = {"value": value, "versions": versions, "evidence": evidence}

    # Seed every project-attribute pair so historical questions always have evidence.
    for project in projects:
        for attribute in M1_ATTRIBUTES:
            conflict = rng.random() < settings["initial_conflict_fraction"]
            initial_count = 2 if conflict else 1
            for version_index in range(initial_count):
                write_fact(
                    project,
                    attribute,
                    update=True,
                    use_alias=(version_index == initial_count - 1)
                    and (rng.random() < settings["alias_probability"]),
                )

    for t in range(settings["T"]):
        project = str(rng.choice(projects))
        attribute = str(rng.choice(M1_ATTRIBUTES))
        append(f"Focus is project {project}. We are reviewing its latest requirements.")
        is_update = rng.random() < settings["update_probability"]
        use_fact_alias = rng.random() < settings["alias_probability"]
        write_fact(project, attribute, update=is_update, use_alias=use_fact_alias)
        append("The next meeting will review progress and remaining questions.")

        if scenario == "stable":
            state = "A"
            p_recent = settings["p_recent_stable"]
        else:
            state = "A" if t < settings["T"] // 2 else "B"
            p_recent = (
                settings["p_recent_abrupt_before"]
                if state == "A"
                else settings["p_recent_abrupt_after"]
            )
        is_recent = rng.random() < p_recent
        if is_recent:
            target_project, target_attribute = project, attribute
            reference = "its"
            qtype = "recent"
        else:
            target_project = str(rng.choice([p for p in projects if p != project]))
            target_attribute = str(rng.choice(M1_ATTRIBUTES))
            reference = target_project
            qtype = "historical"
        question_term = (
            M1_ATTRIBUTE_ALIASES[target_attribute]
            if rng.random() < settings["alias_probability"]
            else target_attribute
        )
        probe_question = f"What is {reference} current {question_term}?"
        target_fact = facts[(target_project, target_attribute)]
        probes.append(
            {
                "t": t,
                "question": probe_question,
                "end": len(history),
                "gold": target_fact["value"],
                "evidence": target_fact["evidence"],
                "old_values": list(target_fact["versions"][:-1]),
                "qtype": qtype,
                "state": state,
            }
        )

    return {
        "seed": seed,
        "scenario": scenario,
        "history": history,
        "probes": probes,
    }


def public_view(episode, probe):
    """Return exactly the current question and its public history prefix."""
    if not isinstance(episode, dict) or not isinstance(probe, dict):
        raise TypeError("episode and probe must be dictionaries")
    end = probe.get("end")
    history = episode.get("history")
    if isinstance(end, bool) or not isinstance(end, int) or not isinstance(history, list):
        raise ValueError("probe.end must be an integer and episode.history a list")
    if not 0 <= end <= len(history):
        raise ValueError("probe.end is outside the episode history")
    return {
        "question": probe["question"],
        "history": [{"id": message["id"], "text": message["text"]} for message in history[:end]],
    }


def audit_truth(episode):
    """Independently reconstruct each answer from parsed public text, not stored gold."""
    episode_errors = []
    history = episode.get("history") if isinstance(episode, dict) else None
    probes = episode.get("probes") if isinstance(episode, dict) else None
    seed = episode.get("seed") if isinstance(episode, dict) else None
    scenario = episode.get("scenario") if isinstance(episode, dict) else None
    if not isinstance(history, list) or not isinstance(probes, list):
        return {
            "seed": seed,
            "scenario": scenario,
            "passed": False,
            "question_count": 0,
            "errors": ["episode must contain history and probes lists"],
            "questions": [],
        }
    expected_ids = list(range(len(history)))
    actual_ids = [message.get("id") if isinstance(message, dict) else None for message in history]
    if actual_ids != expected_ids:
        episode_errors.append("history message IDs are not continuous from zero")
    actual_t = [probe.get("t") if isinstance(probe, dict) else None for probe in probes]
    if actual_t != list(range(len(probes))):
        episode_errors.append("probe t values are not continuous from zero")

    question_checks = []
    for probe in probes:
        errors = []
        t = probe.get("t")
        end = probe.get("end")
        if isinstance(end, bool) or not isinstance(end, int) or not 0 <= end <= len(history):
            question_checks.append(
                {"seed": seed, "scenario": scenario, "t": t, "passed": False,
                 "errors": ["end is outside the available history"]}
            )
            continue
        prefix = history[:end]
        latest_focus = None
        values = {}
        evidence = {}
        versions = {}
        for message in prefix:
            if not isinstance(message, dict) or not isinstance(message.get("text"), str):
                errors.append("visible history contains a malformed message")
                continue
            focus_match = _M1_FOCUS_RE.match(message["text"])
            if focus_match:
                latest_focus = focus_match.group("project")
            parsed = _m1_parse_fact(message["text"])
            if parsed is None:
                continue
            key = (parsed["project"], parsed["attribute"])
            value = parsed["value"]
            values[key] = value
            evidence[key] = message.get("id")
            observed = versions.setdefault(key, [])
            if not observed or observed[-1] != value:
                observed.append(value)

        question = probe.get("question", "")
        if not isinstance(question, str):
            errors.append("question is not text")
            question = ""
        target_project, target_attribute, explicit_project = _m1_parse_question(question, latest_focus)
        if target_project is None:
            errors.append("could not resolve the question project from public history")
        if target_attribute is None:
            errors.append("could not resolve the question attribute")
        key = (target_project, target_attribute)
        derived_gold = values.get(key)
        derived_evidence = evidence.get(key)
        derived_old_values = list(versions.get(key, [])[:-1]) if derived_gold is not None else []
        if derived_gold is None:
            errors.append("no matching fact exists in the visible history prefix")
        if probe.get("gold") != derived_gold:
            errors.append("stored gold does not match the independently parsed latest value")
        if probe.get("evidence") != derived_evidence:
            errors.append("stored evidence does not point to the latest parsed fact message")
        if probe.get("old_values") != derived_old_values:
            errors.append("stored old_values does not match the parsed prior versions")
        stored_evidence = probe.get("evidence")
        if (isinstance(stored_evidence, bool) or not isinstance(stored_evidence, int)
                or not 0 <= stored_evidence < end):
            errors.append("stored evidence is not strictly inside the visible history prefix")
        expected_qtype = "historical" if explicit_project else "recent"
        if target_project is not None and latest_focus is not None:
            expected_qtype = "recent" if target_project == latest_focus else "historical"
        if probe.get("qtype") != expected_qtype:
            errors.append("qtype does not agree with the resolved project and current focus")
        if scenario == "stable":
            expected_state = "A"
        elif scenario == "abrupt":
            expected_state = "A" if isinstance(t, int) and t < len(probes) // 2 else "B"
        else:
            expected_state = None
            errors.append("scenario must be stable or abrupt")
        if expected_state is not None and probe.get("state") != expected_state:
            errors.append("state does not agree with the scenario switch boundary")
        visible = {"question", "history"}
        if set(public_view(episode, probe)) != visible:
            errors.append("public_view exposes fields outside question/history")
        question_checks.append(
            {
                "seed": seed,
                "scenario": scenario,
                "t": t,
                "target_project": target_project,
                "target_attribute": target_attribute,
                "latest_focus": latest_focus,
                "qtype": probe.get("qtype"),
                "state": probe.get("state"),
                "derived_gold": derived_gold,
                "stored_gold": probe.get("gold"),
                "derived_evidence": derived_evidence,
                "stored_evidence": probe.get("evidence"),
                "derived_old_values": derived_old_values,
                "stored_old_values": probe.get("old_values"),
                "history_end": end,
                "passed": not errors,
                "errors": errors,
            }
        )
    passed = not episode_errors and all(row["passed"] for row in question_checks)
    return {
        "seed": seed,
        "scenario": scenario,
        "passed": passed,
        "question_count": len(probes),
        "errors": episode_errors,
        "questions": question_checks,
    }


def _m1_profile(episodes, audit_records):
    rows = []
    for episode, audit in zip(episodes, audit_records):
        probe_checks = {row["t"]: row for row in audit["questions"]}
        history = episode["history"]
        for probe in episode["probes"]:
            check = probe_checks[probe["t"]]
            focus_ids = [
                message["id"] for message in history[:probe["end"]]
                if _M1_FOCUS_RE.match(message["text"])
            ]
            focus_fact = None
            if focus_ids:
                fact_index = focus_ids[-1] + 1
                if fact_index < probe["end"]:
                    focus_fact = _m1_parse_fact(history[fact_index]["text"])
            previous_values = []
            if focus_fact:
                focus_key = (focus_fact["project"], focus_fact["attribute"])
                for message in history[: focus_ids[-1] + 1]:
                    parsed = _m1_parse_fact(message["text"])
                    if parsed and (parsed["project"], parsed["attribute"]) == focus_key:
                        if not previous_values or previous_values[-1] != parsed["value"]:
                            previous_values.append(parsed["value"])
            rows.append(
                {
                    "seed": episode["seed"],
                    "scenario": episode["scenario"],
                    "t": probe["t"],
                    "state": probe["state"],
                    "qtype": probe["qtype"],
                    "has_old_values": bool(probe["old_values"]),
                    "n_old_values": len(probe["old_values"]),
                    "history_messages": probe["end"],
                    "evidence_lag_messages": probe["end"] - 1 - probe["evidence"],
                    "focus_fact_updated": bool(
                        focus_fact and previous_values and previous_values[-1] != focus_fact["value"]
                    ),
                    "focus_fact_uses_alias": bool(
                        focus_fact
                        and focus_fact["attribute"]
                        and history[focus_ids[-1] + 1]["text"].split(": the ", 1)[1].split(" is ", 1)[0]
                        == M1_ATTRIBUTE_ALIASES[focus_fact["attribute"]]
                    ),
                    "target_project": check["target_project"],
                    "target_attribute": check["target_attribute"],
                }
            )
    detail = pd.DataFrame(rows)
    profile = (
        detail.groupby(["scenario", "state"], as_index=False)
        .agg(
            dialogues=("seed", "nunique"),
            questions=("t", "size"),
            recent_fraction=("qtype", lambda s: float((s == "recent").mean())),
            historical_fraction=("qtype", lambda s: float((s == "historical").mean())),
            questions_with_old_values=("has_old_values", "sum"),
            old_value_fraction=("has_old_values", "mean"),
            mean_old_value_count=("n_old_values", "mean"),
            mean_history_messages=("history_messages", "mean"),
            mean_evidence_lag_messages=("evidence_lag_messages", "mean"),
            focus_update_fraction=("focus_fact_updated", "mean"),
            focus_fact_alias_fraction=("focus_fact_uses_alias", "mean"),
            distinct_target_projects=("target_project", "nunique"),
            distinct_target_attributes=("target_attribute", "nunique"),
        )
        .sort_values(["scenario", "state"])
        .reset_index(drop=True)
    )
    return detail, profile


def export_development_data(episodes, run_dir):
    """Export audited development episodes, field dictionary, checks, and profile."""
    run_dir = _m1_Path(run_dir).resolve()
    if "OUTPUTS_DIR" not in globals() or not run_dir.is_relative_to(_m1_Path(OUTPUTS_DIR).resolve()):
        raise ValueError("M1 exports must be written inside outputs/<mode>/<run_id>")
    required_areas = ("data", "tables", "audit")
    if not run_dir.is_dir() or any(not (run_dir / area).is_dir() for area in required_areas):
        raise FileNotFoundError("Initialize the run and its data/tables/audit directories first")
    episodes = list(episodes)
    if not episodes:
        raise ValueError("episodes must not be empty")
    keys = [(ep.get("scenario"), ep.get("seed")) for ep in episodes]
    if len(set(keys)) != len(keys):
        raise ValueError("episode scenario/seed pairs must be unique")
    audit_records = [audit_truth(episode) for episode in episodes]
    if not all(record["passed"] for record in audit_records):
        raise ValueError("Refusing to export episodes that fail independent truth checks")
    detail, profile = _m1_profile(episodes, audit_records)

    data_path = run_dir / "data" / "development_dialogues.jsonl.gz"
    dictionary_path = run_dir / "audit" / "data_dictionary.json"
    checks_path = run_dir / "audit" / "truth_checks.json"
    profile_path = run_dir / "tables" / "development_data_profile.csv"
    targets = (data_path, dictionary_path, checks_path, profile_path)
    existing = [str(path) for path in targets if path.exists()]
    if existing:
        raise FileExistsError("M1 refuses to overwrite existing artifacts: " + ", ".join(existing))

    with _m1_gzip.open(data_path, "xt", encoding="utf-8", newline="\n") as stream:
        for episode in episodes:
            stream.write(
                _m1_json.dumps(episode, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
            )
    save_json(
        dictionary_path,
        {
            "schema_version": "m1-1.0",
            "generator": "ARIN7101 M1 reproducible synthetic template generator",
            "episode_fields": {
                "seed": "Non-negative integer RNG seed for this independent dialogue.",
                "scenario": "stable or abrupt query-demand scenario.",
                "history": "Ordered public messages; message IDs start at zero and are continuous.",
                "probes": "Ordered query records; each end is an exclusive history prefix boundary.",
            },
            "message_fields": {
                "id": "Zero-based message index within the episode.",
                "text": "English synthetic focus, fact, or distractor message.",
            },
            "probe_fields": {
                "t": "Zero-based query index.",
                "question": "Question visible to a strategy.",
                "end": "Exclusive upper bound; public history is history[:end].",
                "gold": "Current value reconstructed from all facts before this query.",
                "evidence": "Message ID of the last supporting statement before this query.",
                "old_values": "Distinct earlier values for the queried project-attribute pair.",
                "qtype": "recent when the question refers to the current focus; historical otherwise.",
                "state": "A/B demand state; only abrupt changes from A to B at T//2.",
            },
            "public_strategy_fields": ["question", "history"],
            "evaluator_only_fields": ["gold", "evidence", "old_values", "qtype", "state", "scenario"],
            "attributes_and_aliases": M1_ATTRIBUTE_ALIASES,
            "message_templates": {
                "focus": "Focus is project Pn. We are reviewing its latest requirements.",
                "fact": "For project Pn: the <attribute or alias> is Vn. <shared system context>",
                "distractor": "The next meeting will review progress and remaining questions.",
            },
            "generation_rules": {
                "projects_per_episode": len(
                    {
                        parsed["project"]
                        for message in episodes[0]["history"]
                        if (parsed := _m1_parse_fact(message["text"])) is not None
                    }
                ),
                "questions_per_episode": sorted({len(episode["probes"]) for episode in episodes}),
                "scenario_switch_index": "T//2; abrupt switches before query t=T//2.",
                "initial_conflicts": "A configured fraction of project-attribute pairs begin with two distinct values.",
                "updates_and_repetitions": "Each round writes a new value with update_probability; otherwise repeats the current value.",
                "latest_evidence": "The latest statement is evidence even when it repeats the current value.",
                "external_data_or_llm": False,
            },
            "statistical_unit": "independent dialogue; queries within a dialogue are dependent",
            "episodes": [
                {"seed": episode["seed"], "scenario": episode["scenario"], "questions": len(episode["probes"])}
                for episode in episodes
            ],
        },
    )
    save_json(
        checks_path,
        {
            "status": "passed",
            "episode_count": len(episodes),
            "question_count": sum(record["question_count"] for record in audit_records),
            "all_questions_independently_reconstructed": True,
            "episodes": audit_records,
        },
    )
    save_csv(profile_path, profile)
    return {
        "run_dir": str(run_dir),
        "episodes": len(episodes),
        "questions": int(sum(record["question_count"] for record in audit_records)),
        "detail_profile": detail,
        "profile": profile,
        "files": [str(path) for path in targets],
    }


In [ ]:
import gzip as _m1_test_gzip
import json as _m1_test_json
import tempfile as _m1_test_tempfile
from pathlib import Path as _m1_test_Path

M1_TEST_CONFIG = {
    "T": 8,
    "n_projects": 3,
    "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9,
    "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25,
    "initial_conflict_fraction": 0.25,
    "alias_probability": 0.5,
}

M1_FIXTURE = {
    "seed": 7,
    "scenario": "stable",
    "history": [
        {"id": 0, "text": "Focus is project P1. We are reviewing its latest requirements."},
        {"id": 1, "text": "For project P1: the dataset is V1. " + M1_SYSTEM_CONTEXT},
        {"id": 2, "text": "For project P1: the data source is V2. " + M1_SYSTEM_CONTEXT},
        {"id": 3, "text": "The next meeting will review progress and remaining questions."},
        {"id": 4, "text": "Focus is project P2. We are reviewing its latest requirements."},
        {"id": 5, "text": "For project P2: the language is V3. " + M1_SYSTEM_CONTEXT},
        {"id": 6, "text": "For project P1: the dataset is V2. " + M1_SYSTEM_CONTEXT},
    ],
    "probes": [
        {
            "t": 0,
            "question": "What is its current data source?",
            "end": 4,
            "gold": "V2",
            "evidence": 2,
            "old_values": ["V1"],
            "qtype": "recent",
            "state": "A",
        },
        {
            "t": 1,
            "question": "What is P1 current dataset?",
            "end": 6,
            "gold": "V2",
            "evidence": 2,
            "old_values": ["V1"],
            "qtype": "historical",
            "state": "A",
        },
        {
            "t": 2,
            "question": "What is P1 current dataset?",
            "end": 7,
            "gold": "V2",
            "evidence": 6,
            "old_values": ["V1"],
            "qtype": "historical",
            "state": "A",
        },
    ],
}

_m1_test_stable_a = generate_episode(31415, "stable", M1_TEST_CONFIG)
_m1_test_stable_b = generate_episode(31415, "stable", M1_TEST_CONFIG)
assert _m1_test_stable_a == _m1_test_stable_b, "same seed/config must reproduce the episode"
assert _m1_test_stable_a != generate_episode(31416, "stable", M1_TEST_CONFIG)
assert [m["id"] for m in _m1_test_stable_a["history"]] == list(
    range(len(_m1_test_stable_a["history"]))
)
assert [p["t"] for p in _m1_test_stable_a["probes"]] == list(range(8))
assert audit_truth(_m1_test_stable_a)["passed"]

_m1_test_abrupt = generate_episode(31415, "abrupt", M1_TEST_CONFIG)
assert [p["state"] for p in _m1_test_abrupt["probes"]] == ["A"] * 4 + ["B"] * 4
assert audit_truth(_m1_test_abrupt)["passed"]

_m1_test_probe = _m1_test_stable_a["probes"][0]
_m1_test_view = public_view(_m1_test_stable_a, _m1_test_probe)
assert set(_m1_test_view) == {"question", "history"}
assert len(_m1_test_view["history"]) == _m1_test_probe["end"]
assert all(set(m) == {"id", "text"} for m in _m1_test_view["history"])
_m1_test_future_variant = _m1_test_json.loads(_m1_test_json.dumps(_m1_test_stable_a))
_m1_test_future_variant["history"][_m1_test_probe["end"]]["text"] = "Future-only mutation."
assert public_view(_m1_test_future_variant, _m1_test_future_variant["probes"][0]) == _m1_test_view

_m1_test_fixture_audit = audit_truth(M1_FIXTURE)
assert _m1_test_fixture_audit["passed"]
assert _m1_test_fixture_audit["questions"][0]["derived_gold"] == "V2"
assert _m1_test_fixture_audit["questions"][0]["derived_evidence"] == 2
assert _m1_test_fixture_audit["questions"][2]["derived_evidence"] == 6, (
    "a repeated value must still move evidence to its latest statement"
)

_m1_test_corrupt = _m1_test_json.loads(_m1_test_json.dumps(M1_FIXTURE))
_m1_test_corrupt["probes"][0]["gold"] = "V999"
_m1_test_corrupt["probes"][0]["evidence"] = 1
_m1_test_corrupt["probes"][0]["old_values"] = []
_m1_test_corrupt_audit = audit_truth(_m1_test_corrupt)
assert not _m1_test_corrupt_audit["passed"]
assert len(_m1_test_corrupt_audit["questions"][0]["errors"]) >= 3
_m1_test_future_evidence = _m1_test_json.loads(_m1_test_json.dumps(M1_FIXTURE))
_m1_test_future_evidence["probes"][0]["evidence"] = 6
_m1_future_audit = audit_truth(_m1_test_future_evidence)
assert any("strictly inside" in error for error in _m1_future_audit["questions"][0]["errors"])

try:
    generate_episode(1, "unsupported", M1_TEST_CONFIG)
except ValueError:
    pass
else:
    raise AssertionError("unsupported scenarios must fail clearly")

with _m1_test_tempfile.TemporaryDirectory(prefix="m1_fixture_", dir=OUTPUTS_DIR) as _m1_temp:
    _m1_test_run = _m1_test_Path(_m1_temp)
    for _m1_area in ("data", "tables", "audit"):
        (_m1_test_run / _m1_area).mkdir()
    _m1_export = export_development_data([M1_FIXTURE], _m1_test_run)
    assert _m1_export["episodes"] == 1 and _m1_export["questions"] == 3
    _m1_data_path = _m1_test_run / "data" / "development_dialogues.jsonl.gz"
    with _m1_test_gzip.open(_m1_data_path, "rt", encoding="utf-8") as _m1_stream:
        _m1_roundtrip = _m1_test_json.loads(_m1_stream.readline())
    assert _m1_roundtrip == M1_FIXTURE
    try:
        export_development_data([M1_FIXTURE], _m1_test_run)
    except FileExistsError:
        pass
    else:
        raise AssertionError("M1 exporter must refuse to overwrite existing artifacts")
    assert len(list(_m1_test_run.rglob("*"))) >= 7

M1_TEST_RESULTS = {
    "status": "passed",
    "same_seed_reproducibility": True,
    "different_seed_changes_episode": True,
    "continuous_ids_and_time": True,
    "abrupt_switch_at_T_half": True,
    "public_view_hides_evaluator_fields_and_future": True,
    "independent_parser_handles_its_alias_explicit_project_and_versions": True,
    "repeated_value_moves_latest_evidence": True,
    "audit_detects_corrupt_gold_evidence_and_old_values": True,
    "audit_rejects_future_evidence": True,
    "export_roundtrip_and_no_overwrite": True,
    "fixture_export_is_temporary_and_excluded_from_development_data": True,
}
print("M1 checks passed:", _m1_test_json.dumps(M1_TEST_RESULTS, sort_keys=True))

# Set this True only after choosing a new development RUN_ID in the shared paths cell.
M1_EXPORT_DEVELOPMENT = False
if M1_EXPORT_DEVELOPMENT:
    if MODE != "development" or RUN_ID == "local_setup":
        raise RuntimeError("Choose MODE='development' and a new, explicit M1 RUN_ID first")
    _m1_dev_episodes = [
        generate_episode(int(PROTOCOL_DRAFT["dev_seed"]) + 100 * scenario_index + index,
                         scenario, PROTOCOL_DRAFT)
        for scenario_index, scenario in enumerate(SCENARIOS)
        for index in range(int(PROTOCOL_DRAFT["n_dev_per_scenario"]))
    ]
    M1_DEVELOPMENT_EXPORT = export_development_data(_m1_dev_episodes, RUN_DIR)
    print("M1 development export:", M1_DEVELOPMENT_EXPORT["run_dir"])


### M1研究记录（自动化实现后待成员复核）

本轮按共同草案实现6个项目、4种属性和每对话300题；更新、复述、别名和初始多版本规则见 `deliverables/M1_notes.md`。独立解析器从每题公开历史前缀重建值与证据，不使用保存的 `gold` 作为推导来源。

Development run `outputs/development/m1_development_20261109` 包含6条对话、1,800题；1,800题自动真值核查通过，gzip数据读回一致。实测近期题比例：stable/A 0.890、abrupt/A 0.893、abrupt/B 0.102；后半段旧值比例0.898，平均证据距离72.362条消息。比例是随机样本观察，不是策略效果结论。

自动测试覆盖seed复现、连续ID、信息前缀、切换边界、更新/复述/别名、真值篡改检测和无覆盖导出。测试通过不等于人工审查或团队验收。

M2四臂材料和M1/M2十题双签尚未完成；M2/M4接收待确认。姓名留待成员本人填写。代码和记录由Codex按项目负责人的授权协助完成，成员需亲自复跑、理解并确认AI辅助说明后再署名。


<a id="member-m2"></a>
## M2 · 策略、回答器与成本
**四臂选择、规则回答与评价代码由你编写。任务区间m2-*。**
输入：M1公开视图与episode；上游未完成时自建短历史。输出给M3/M4，审计材料交M1。

1. **定义公平比较。** 研究共同输入模板、计数规则、UNKNOWN与最新事实优先规则；四臂必须共享回答器与成本口径。
2. **逐臂实现。** 从连续窗口到TF–IDF，再实现仅用公开历史的指代消解和同义词处理；记录平局、超预算消息、focus锚点处理。
3. **写自己的测试。** 检查预算边界、空历史、未来隔离、时序、旧值；不可读取gold/隐藏题型。
4. **评价与研究。** 为每题每臂导出行，记录完整输入成本和候选构造耗时，研究accuracy/coverage/outdated的关系；结果由实验决定。
5. **交接。** 导出10题的公开历史、各臂上下文/回答/成本和评估真值；M1独立追踪，你交叉复核。先给M3单对话表，再给M4全部开发表。

**验收方向：** 预算只管上下文；C含共同指令、上下文、问题；id唯一且时序；完整/空历史行为有依据。至少三个自己发现或构造的边界案例及测试局限。
**输出：** tables/development_outcomes.csv、strategy_by_query.csv；audit/audit_examples.json、audit_review.csv、strategy_checks.json。人工审计不能自动填pass。


In [ ]:
def count_input_units(text):
    """M2实现：统一长度代理，不声称是真实token。"""
    raise NotImplementedError("M2：实现成本计数")

def select_context(view, arm, cfg):
    """M2实现：返回预算内、id升序的消息列表；view只有question/history。"""
    raise NotImplementedError("M2：实现四策略")

def rule_reader(question, context):
    """M2实现：仅依所给context回答；缺必要证据返回UNKNOWN。"""
    raise NotImplementedError("M2：实现回答器")

def evaluate_episode(episode, cfg):
    """M2实现：返回四臂评价DataFrame，评估真值不泄漏给策略。"""
    raise NotImplementedError("M2：实现评价和指标")

def export_strategy_evidence(episodes, outcomes, run_dir):
    """M2实现：导出开发表、10题审计与检查证据；保留人工待审状态。"""
    raise NotImplementedError("M2：实现证据导出")


In [ ]:
# TODO M2：自建恰好预算/超预算、重复事实、旧值、缺focus和空历史案例。
# 设计当前公开前缀相同、未来不同的输入，检验未来隔离。
# 各臂单测后再与M1联调；添加自己的开发统计与审计导出调用。


### M2研究记录（本人填写）
四臂实际权衡是否符合预期？找出coverage与accuracy不同的证据。为什么不能用proxy成本声称真实API节省？
具体seed/t/arm、接口版本、M1/M3/M4接收记录：待填写。


<a id="member-m3"></a>
## M3 · 贝叶斯建模与回放
**后验更新、决策、优化、回放与数值验证由你实现。任务区间m3-*。**
输入：问题、已知四候选cost、M2单对话评价表；输出：统一学习器、回放、推导与核查；M4接收。

1. **先推导。** 从Bernoulli似然/Beta先验推导TS后验，手算收到1、1、0后的参数与均值；解释成本为什么不应计为失败次数。
2. **实现TS与参照。** 定义状态、采样和成本决策；实现四固定臂与基于公开问题的RuleRouter，研究先验和探索。
3. **研究CTS。** 每臂贝叶斯logistic与高斯先验；自己推导MAP目标、梯度、Hessian、Laplace协方差。实现优化、采样、每5次已选反馈重拟合；解释特征共线性与数值稳定性。
4. **实现回放。** 每新对话新建学习器；choose只看q/costs，update只看已选臂0/1反馈；全臂表仅属于评估器。
5. **验证交付。** 自己写手算一致、有限差分、协方差、重置、反馈隔离案例；报告拟合失败，不静默吞掉。交M4微型输入/输出及M2真实表上的检查。

**研究问题：** 如何检测未选奖励泄漏？同seed是否复现？特征如何不借用隐藏标签？TS与CTS差异是否足以构成严格特征消融？
**输出：** audit/method_checks.json、tables/development_replay.csv；推导和局限写M3_notes。


In [ ]:
def question_features(question):
    """M3实现：三维[截距, 显式项目名, its]，只读取问题。"""
    raise NotImplementedError("M3：实现公开特征")

class Selector:
    def __init__(self, method, seed, cfg):
        """M3实现：支持固定臂、TS、LaplaceCTS、RuleRouter，初始化状态。"""
        raise NotImplementedError("M3：设计学习器状态")

    def choose(self, question, costs):
        """M3实现：返回ARMS索引，不可读取当前真实奖励。"""
        raise NotImplementedError("M3：实现决策")

    def update(self, arm_index, reward, question):
        """M3实现：reward仅为已选臂0/1正确性反馈。"""
        raise NotImplementedError("M3：实现后验更新")

def replay_episode(arm_outcomes, method, algorithm_seed, cfg):
    """M3实现：单对话全臂表→该方法逐题回放DataFrame。"""
    raise NotImplementedError("M3：实现信息隔离的回放")


In [ ]:
# TODO M3：Beta手算对照、MAP梯度/Hessian有限差分。
# 第5次反馈前后、协方差有效性、新对话重置。
# 设计能暴露未选奖励泄漏的测试，而非只检查函数签名。
# 用自建小表再用M2交付表；保留失败数值与修复说明。


### M3研究记录（本人填写）
推导如何对应自己的代码？拟合失败如何处理？哪些是测试性质、哪些是建模假设？什么条件下方法会失效？
M4接收的接口、检查证据和方法说明：待填写。不要预写CTS更好。


<a id="member-m4"></a>
## M4 · 集成、实验设计与统计
**你要编写调度、开发选择、冻结、统计和归档。任务区间m4-*。**
输入：M1–M3实现、证据、审查；输出：真实run、表、manifest和判断；M3复核、M5接收。

1. **先做统计手算。** 自建少量对话小表验证配对和算法seed的对话内平均；这是fixture，不能作为研究结果。
2. **接通demo。** 编写入口逐模块调用；按场景仅用开发平均效用选固定基线，说明平局规则，不用测试结果选基线。
3. **实现冻结与失败管理。** 核验M1/M2审计、M3方法检查、全组确认；保存配置/源码/证据快照、哈希、版本，检查seed分离、防覆盖；登记每次正式尝试和已看数据情况。
4. **正式调度。** 每场景30条×300题，四臂评价与方法回放；新对话重置，两个算法seed不是两份独立数据。先查完整性再看效果。
5. **自行实现统计与判断。** 对话级配对bootstrap，主次比较、95%及两主比较97.5%区间；区间含0不等于等效，点估计到0.02不等于下界超过0.02。
6. **可复现交付。** 导出同run完整表和manifest，让M3复核聚合/配对/区间，M5独立读取。保留负结果，不换seed直到显著。

**输出：** data/evaluation_dialogues.jsonl.gz；tables/evaluation_outcomes.csv、replay.csv、episode_metrics.csv、primary_effects.csv、secondary_effects.csv、method_summary.csv；config/frozen_config.json、notebook_snapshot.ipynb；audit/review_snapshot.json；reports/M4_handoff.md；根级manifest.json。实现与产物均待你完成。


In [ ]:
def choose_fixed_baselines(development_outcomes):
    """M4实现：只用开发平均效用，记录平局规则。"""
    raise NotImplementedError("M4：实现开发选择")

def freeze_protocol(cfg, baselines, evidence_paths, run_dir):
    """M4实现：核验交接、保存冻结快照/seed记录，在测试数据生成前调用。"""
    raise NotImplementedError("M4：实现冻结与防覆盖")

def run_evaluation(episodes, methods, algorithm_seeds, cfg):
    """M4实现：调用M2/M3，返回全臂表/回放表，核验规模与唯一键。"""
    raise NotImplementedError("M4：实现实验集成")

def aggregate_dialogues(replay_table):
    """M4实现：对话级指标；算法seed不视为新增独立样本。"""
    raise NotImplementedError("M4：实现正确聚合")

def paired_effects(episode_metrics, baselines, cfg):
    """M4实现：配对重采样独立对话，输出主次效应表。"""
    raise NotImplementedError("M4：实现bootstrap与判断")

def export_run(run_dir, artifacts, metadata):
    """M4实现：按契约保存结果/manifest供M5独立验证。"""
    raise NotImplementedError("M4：实现归档")


In [ ]:
# TODO M4：先写手算对照，再实现调度、配对与聚合。
# 检查dev/test seed不相交、轨迹无缺失/重复、算法seed齐全。
# 研究错误地以题为单位重采样会有何影响。
# 接口验收后自己增加development/demo入口；正式入口先核验冻结。
# 未满足条件明确停止，不填假结果，不自动替成员签名。


### M4结果判断（实际实验后本人填写）
主比较、基线选择依据、统计单位和n；效应方向/区间/0.02含义；负结果、异常、修正与已看情况；M3复核；交付run/commit/manifest校验和：待填写。未完成前只写计划。


<a id="member-m5"></a>
## M5 · 可视化研究、结果解释与汇报
**图表代码、读取校验与评价由你完成。任务区间m5-*。**
输入：M4完整归档及M1–M3说明；输出：自己设计的图、解读与汇报；全组复核。

1. **不等结果开始。** 设计叙事与每图的问题，用明确标为fixture的小表探索图型/颜色/误差线。最终不能混入假数据。
2. **实现独立读取。** 指定run路径，校验manifest、文件、模式、配置和规模，不依赖M4内存，不自动取“最新目录”；和M4共同验收契约。
3. **自主设计与编码。** 下表是问题菜单，不是画法答案。解释图型、聚合和区间选择；可协商合并或调整图数，不为凑数重复展示。
4. **自己评估。** 对照CSV复核数值，每图写观察→证据→局限，与对应成员核对。不显著或方向相反也如实展示，不能复制参考本结论。
5. **完成汇报。** M1背景、M2策略、M3方法、M4统计、M5叙事；旧计划约15–16页slides/15–20分钟视频，须核对课程要求，说明个人贡献与AI辅助。

| 建议问题 | 需要的数据 | 联合复核 |
|---|---|---|
| 主效应是否明确？ | primary/secondary_effects | M4/M3 |
| 准确率、成本、效用如何取舍？ | episode_metrics/method_summary | M2 |
| 效用差由什么组成？ | 同一基线的分量差 | M2/M4 |
| 不同需求怎样选臂？ | replay及题型 | M3/M1 |
| 收益是否贯穿对话？ | 配对逐题轨迹 | M4 |
| 策略有何互补/失败机制？ | 全臂表与具体例子 | M1/M2 |
| 突变前后是否相同？ | 阶段对话配对表 | M4 |

**边界：** 描述性区间不增加主检验；逐点带不是同时置信带；固定基线累计差不是理论regret；proxy成本不是实际费用。
**输出：** figures/<figure_id>.png/.svg、tables/figure_<figure_id>.csv、reports/figure_interpretations.md、reports/figure_manifest.json。只新增图表文件，不改M4数据；重绘使用新figure_id或明确render子目录并记录来源。


In [ ]:
def load_verified_run(run_dir):
    """M5实现：独立校验归档后读取绘图所需数据。"""
    raise NotImplementedError("M5：实现来源校验和读取")

def build_figures(results, cfg):
    """M5实现：自行选择图型、聚合/误差线，返回图对象和对应数据表。"""
    raise NotImplementedError("M5：设计并实现可视化")

def export_visual_evidence(figures, figure_tables, source_run, render_id):
    """M5实现：保存图/表/来源清单，不修改上游原始数据。"""
    raise NotImplementedError("M5：实现图表交付")


In [ ]:
# TODO M5：用fixture先检查绘图；模拟缺文件/改CSV/混run验证读取器。
# 新内核只执行共享单元与M5模块，不依赖M4变量。
# 检查数据-图一致、单位/区间、负值标签、图例。
# 实现后增加绘图调用；本人写解读，不用固定文字自动宣布胜出。


### M5逐图研究记录（本人填写）
每图的问题、来源run与表、汇总方式、观察、支持/不支持的结论、局限、复核人：待填写。
随后补slides、分工、视频链接与提交清单。未有结果时只写计划与示意图设计。


## 完成标准
todo → doing → review → done：自己的实现与研究说明完成，自己设计的案例通过，接收人能按接口使用并复核后再标done。结构检查不证明算法正确或项目完成。

正式评估前冻结共同协议；看过测试结果之后的探索、修正和失败必须留记录，不能反复换seed至显著。

课程线索：Lecture 2的Beta/MAP/Laplace，Lecture 6的bandit/后验采样，Lecture 4的开发/评估隔离，Lecture 1的建模与不确定性。成员自行阅读原课件、推导和引用，本工作本不提供完成的推导答案。

模块完成后由M4自行编写集成入口；各成员用自己的测试、实验与交叉复核证明成果。
